### Odds API Basics


This code defines some function to be used to query the Odds API.  It can query for a particualr sport and market and automatically write those odds to a file.  It has a bundled function to collect all 3 marketsm; moneyline , spread, and totals, with a single call.

In [ ]:
import requests
import pandas as pd


# Get a free API key at https://api.the-odds-api.com/
API_KEY = '24a58d9f2f2e9a780e9459315f20f7ca'

REGIONS = 'us' # uk | us | eu | au. Multiple can be specified if comma delimited

MARKETS = 'h2h' # h2h | spreads | totals. Multiple can be specified if comma delimited

ODDS_FORMAT = 'decimal' # decimal | american

DATE_FORMAT = 'iso' # iso | unix

SPORT = "americanfootball_nfl"

In [ ]:
# This code executes a non-metered function to get the availbale spor

sports_response = requests.get(
    'https://api.the-odds-api.com/v4/sports', 
    params={'api_key': API_KEY}
)

if sports_response.status_code != 200:
    print(f'Failed to get sports: status_code {sports_response.status_code}, response body {sports_response.text}')
else:
    sports_data = sports_response.json()  # This is a list of dicts
    sports_data_df = pd.DataFrame(sports_data)        # Convert to DataFrame
    print(sports_data_df.head())                      # Show first few rows


In [ ]:
# This set of function is used to get current oods from the Odds API

import os
import requests
import pandas as pd
from datetime import datetime, timezone


def fetch_odds_json(
        
    # This is a low level function to get the odds json data from the Odds API

    api_key: str,
    sport: str = "americanfootball_nfl",
    regions: str = "us",
    markets: str = "h2h",
    odds_format: str = "decimal",
    date_format: str = "iso",
):
    """Low-level helper: call The Odds API and return (json_data, response_headers)."""
    base_url = "https://api.the-odds-api.com/v4/sports"

    params = {
        "api_key": api_key,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": date_format,
    }

    resp = requests.get(f"{base_url}/{sport}/odds", params=params, timeout=30)

    if resp.status_code != 200:
        raise RuntimeError(
            f"Failed to get odds: status_code={resp.status_code}, body={resp.text}"
        )

    return resp.json(), resp.headers


def flatten_odds_to_df(odds_json) -> pd.DataFrame:

    """Flatten Odds API JSON into one row per game–bookmaker–market–outcome."""

    # This is a helper function to flatten the nested json data into a flat dataframe
    # It is generic in the sense that it does not assume specific markets or outcomes
    rows = []

    for game in odds_json:
        game_meta = {
            "game_id": game.get("id"),
            "sport_key": game.get("sport_key"),
            "sport_title": game.get("sport_title"),
            "commence_time": game.get("commence_time"),
            "home_team": game.get("home_team"),
            "away_team": game.get("away_team"),
        }

        for bm in game.get("bookmakers", []):
            bm_meta = {
                "bookmaker_key": bm.get("key"),
                "bookmaker_title": bm.get("title"),
                "bookmaker_last_update": bm.get("last_update"),
            }

            for market in bm.get("markets", []):
                market_meta = {
                    "market_key": market.get("key"),
                    "market_last_update": market.get("last_update"),
                }

                for outcome in market.get("outcomes", []):
                    # Generic unpack: includes price, point, etc.
                    row = {**game_meta, **bm_meta, **market_meta, **outcome}
                    rows.append(row)

    return pd.DataFrame(rows)


def get_odds_df(
        
    # This function will get the odds for a particular sport and market, flatten to a dataframe, and optionally save to csv

    api_key: str,
    sport: str = "americanfootball_nfl",
    regions: str = "us",
    markets: str = "h2h",
    odds_format: str = "decimal",
    date_format: str = "iso",
    save_file: bool = True,
    save_path: str = r"C:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\Code\OddsExports\\",
) -> pd.DataFrame:
    """
    Fetch odds data for one or more markets, flatten to DataFrame, optionally save to CSV.
    """
    odds_json, headers = fetch_odds_json(
        api_key=api_key,
        sport=sport,
        regions=regions,
        markets=markets,
        odds_format=odds_format,
        date_format=date_format,
    )

    if not odds_json:
        print("No events returned from API.")
        return pd.DataFrame()

    df = flatten_odds_to_df(odds_json)

    # Quota info
    remaining = headers.get("x-requests-remaining")
    used = headers.get("x-requests-used")
    if remaining is not None or used is not None:
        print(f"Requests remaining: {remaining}, used: {used}")

    # Save file
    if save_file and not df.empty:
        ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        safe_sport = sport.replace("/", "-")
        safe_markets = markets.replace(",", "-")
        os.makedirs(save_path, exist_ok=True)
        filename = os.path.join(
            save_path, f"odds_{safe_sport}_{safe_markets}_{ts}.csv"
        )
        df.to_csv(filename, index=False)
        print(f"Saved {len(df)} rows to {filename}")

    return df


def get_all_markets_odds(
        
     # This function will get the odds for all 3 standard markets and return a dictionary of dataframes   
    api_key: str,
    sport: str = "americanfootball_nfl",
    regions: str = "us",
    odds_format: str = "decimal",
    date_format: str = "iso",
    save_file: bool = True,
    save_path: str = r"C:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\Code\OddsExports\\",
):
    """
    Wrapper that loops through the 3 standard markets: h2h, spreads, totals.
    Returns a dictionary of DataFrames keyed by market name.
    """
    all_dfs = {}
    for market in ["h2h", "spreads", "totals"]:
        print(f"\nFetching {market} data...")
        df = get_odds_df(
            api_key=api_key,
            sport=sport,
            regions=regions,
            markets=market,
            odds_format=odds_format,
            date_format=date_format,
            save_file=save_file,
            save_path=save_path,
        )
        all_dfs[market] = df
    return all_dfs


In [ ]:


REGIONS = "us"
MARKETS = "h2h"
ODDS_FORMAT = "decimal"
DATE_FORMAT = "iso"
SPORT = "americanfootball_nfl"

odds_df = get_odds_df(
    api_key=API_KEY,
    sport=SPORT,
    regions=REGIONS,
    markets=MARKETS,
    save_file=True
)

print(odds_df.head())


In [ ]:
odds_df = get_odds_df(
    api_key=API_KEY,
    sport=SPORT,
    regions=REGIONS,
    markets="spreads",
    save_file=True
)


In [ ]:
odds_df = get_odds_df(
    api_key=API_KEY,
    sport=SPORT,
    regions=REGIONS,
    markets="totals",
    save_file=True
)



In [ ]:
all_odds = get_all_markets_odds(api_key=API_KEY, sport= "americanfootball_nfl")

In [ ]:
all_odds = get_all_markets_odds(api_key=API_KEY, sport= "americanfootball_ncaaf")

In [ ]:
all_odds = get_all_markets_odds(api_key=API_KEY, sport= "americanfootball_ncaaf")  

In [ ]:
all_odds = get_all_markets_odds(api_key=API_KEY, sport= "basketball_nba")  